# 03 — Inference, Benchmarking & Deployment

Notebook này:
1. Load trained VideoMAEv2-Small model
2. Benchmark inference speed (latency, throughput)
3. Test predictions trên video samples
4. Hướng dẫn deploy Streamlit app

**Target**: GPU 4GB VRAM local (inference ~100-150ms/video)

In [ ]:
# ============================================================
# CẤU HÌNH
# ============================================================

NUM_CLASSES = 50
NUM_FRAMES = 16
IMAGE_SIZE = 224
BENCHMARK_RUNS = 50  # Number of runs for latency measurement

In [ ]:
import os
import sys
import json
import time
import numpy as np
from pathlib import Path

import torch
import torch.nn as nn
from transformers import VideoMAEForVideoClassification
from torchvision.transforms import Resize, CenterCrop, Normalize
import matplotlib.pyplot as plt

# Detect environment
def detect_environment():
    try:
        import google.colab
        return "colab"
    except ImportError:
        pass
    if os.path.exists("/kaggle/working"):
        return "kaggle"
    return "local"

ENV = detect_environment()
if ENV == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = "/content/drive/MyDrive/vsl-recognition"
elif ENV == "kaggle":
    BASE_DIR = "/kaggle/working/vsl-recognition"
else:
    BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

MODEL_DIR = os.path.join(BASE_DIR, "models")
DATA_DIR = os.path.join(BASE_DIR, "data", "multi_vsl")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Environment: {ENV}")
print(f"🔧 Device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"   VRAM: {gpu_mem:.1f} GB")

## 1. Load Trained Model

In [ ]:
# Load trained model
model_path = os.path.join(MODEL_DIR, "videomae_vsl_best")

if not os.path.exists(model_path):
    print(f"⚠️ Model not found at: {model_path}")
    print(f"   Please run notebook 02 first to train the model.")
    print(f"   Or download a pre-trained model to this path.")
else:
    model = VideoMAEForVideoClassification.from_pretrained(
        model_path, num_labels=NUM_CLASSES
    )
    model = model.to(device)
    model.eval()
    
    # Load class names
    with open(os.path.join(model_path, "class_names.json")) as f:
        class_names = json.load(f)
    
    total_params = sum(p.numel() for p in model.parameters())
    print(f"✅ Model loaded from: {model_path}")
    print(f"   Params: {total_params / 1e6:.1f}M")
    print(f"   Classes: {len(class_names)}")
    print(f"   Class names: {class_names[:10]}...")
    
    if torch.cuda.is_available():
        mem_used = torch.cuda.memory_allocated() / 1e9
        print(f"   VRAM used: {mem_used:.2f} GB")

## 2. Inference Speed Benchmark

Đo latency trên GPU local (hoặc Colab T4) để verify đáp ứng yêu cầu realtime.

In [ ]:
@torch.no_grad()
def benchmark_inference(model, device, num_frames=16, image_size=224, num_runs=50):
    """Benchmark model inference latency."""
    # Create dummy input
    dummy_input = torch.randn(1, num_frames, 3, image_size, image_size).to(device)
    
    # Warmup
    for _ in range(5):
        _ = model(pixel_values=dummy_input)
    
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    
    # Benchmark
    latencies = []
    for _ in range(num_runs):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        start = time.perf_counter()
        
        _ = model(pixel_values=dummy_input)
        
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        end = time.perf_counter()
        latencies.append((end - start) * 1000)  # ms
    
    latencies = np.array(latencies)
    
    # Also test with fp16 if GPU available
    fp16_latencies = None
    if torch.cuda.is_available():
        model_fp16 = model.half()
        dummy_fp16 = dummy_input.half()
        
        for _ in range(5):
            _ = model_fp16(pixel_values=dummy_fp16)
        torch.cuda.synchronize()
        
        fp16_lats = []
        for _ in range(num_runs):
            torch.cuda.synchronize()
            start = time.perf_counter()
            _ = model_fp16(pixel_values=dummy_fp16)
            torch.cuda.synchronize()
            end = time.perf_counter()
            fp16_lats.append((end - start) * 1000)
        
        fp16_latencies = np.array(fp16_lats)
        model.float()  # Restore
    
    return latencies, fp16_latencies


print("⏱️ Running inference benchmark...\
")
latencies_fp32, latencies_fp16 = benchmark_inference(
    model, device, NUM_FRAMES, IMAGE_SIZE, BENCHMARK_RUNS
)

print(f"📊 Inference Latency (FP32):")
print(f"   Mean: {latencies_fp32.mean():.1f} ms")
print(f"   Median: {np.median(latencies_fp32):.1f} ms")
print(f"   Std: {latencies_fp32.std():.1f} ms")
print(f"   P95: {np.percentile(latencies_fp32, 95):.1f} ms")
print(f"   FPS: {1000 / latencies_fp32.mean():.1f}")

if latencies_fp16 is not None:
    print(f"\
📊 Inference Latency (FP16):")
    print(f"   Mean: {latencies_fp16.mean():.1f} ms")
    print(f"   Median: {np.median(latencies_fp16):.1f} ms")
    print(f"   FPS: {1000 / latencies_fp16.mean():.1f}")
    print(f"   Speedup: {latencies_fp32.mean() / latencies_fp16.mean():.2f}x")

# VRAM usage
if torch.cuda.is_available():
    print(f"\
💾 VRAM Usage:")
    print(f"   Model: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"   Peak (during inference): {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")
    print(f"   Available: {(torch.cuda.get_device_properties(0).total_mem - torch.cuda.memory_allocated()) / 1e9:.2f} GB")

## 3. Test on Real Videos

In [ ]:
import cv2

def load_video(video_path: str, num_frames: int = 16) -> np.ndarray:
    """Load video frames."""
    try:
        from decord import VideoReader, cpu
        vr = VideoReader(video_path, ctx=cpu(0))
        total = len(vr)
        if total >= num_frames:
            indices = np.linspace(0, total - 1, num_frames, dtype=int)
        else:
            indices = np.concatenate([np.arange(total), np.full(num_frames - total, total - 1, dtype=int)])
        return vr.get_batch(indices).asnumpy()
    except Exception:
        cap = cv2.VideoCapture(video_path)
        frames = []
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        cap.release()
        frames = np.array(frames)
        total = len(frames)
        if total >= num_frames:
            indices = np.linspace(0, total - 1, num_frames, dtype=int)
        else:
            indices = np.concatenate([np.arange(total), np.full(num_frames - total, total - 1, dtype=int)])
        return frames[indices]


@torch.no_grad()
def predict_video(model, video_path: str, class_names: list, device, num_frames=16):
    """Full inference pipeline: video file → prediction."""
    start = time.perf_counter()
    
    # Load
    frames = load_video(video_path, num_frames)
    
    # Transform
    video = torch.from_numpy(frames).float() / 255.0
    video = video.permute(0, 3, 1, 2)  # (T, 3, H, W)
    
    normalize = Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    transformed = []
    for t in range(num_frames):
        frame = Resize(IMAGE_SIZE + 32, antialias=True)(video[t])
        frame = CenterCrop(IMAGE_SIZE)(frame)
        frame = normalize(frame)
        transformed.append(frame)
    
    input_tensor = torch.stack(transformed).unsqueeze(0).to(device)
    
    # Predict
    outputs = model(pixel_values=input_tensor)
    probs = torch.softmax(outputs.logits[0], dim=0)
    top5_probs, top5_idx = torch.topk(probs, 5)
    
    latency = (time.perf_counter() - start) * 1000
    
    return {
        "label": class_names[top5_idx[0].item()],
        "confidence": top5_probs[0].item(),
        "top5": [(class_names[i.item()], p.item()) for i, p in zip(top5_idx, top5_probs)],
        "latency_ms": latency,
        "frames": frames,
    }


# Test on sample videos
data_path = Path(DATA_DIR)
all_videos = list(data_path.rglob("*.avi")) + list(data_path.rglob("*.mp4"))

if all_videos:
    import random
    random.seed(42)
    samples = random.sample(all_videos, min(6, len(all_videos)))
    
    print("🎯 Testing on sample videos:\
")
    for video_path in samples:
        result = predict_video(model, str(video_path), class_names, device)
        true_label = video_path.parent.name
        correct = "✅" if result["label"] == true_label else "❌"
        
        print(f"  {correct} True: {true_label:15s} | Pred: {result['label']:15s} | "
              f"Conf: {result['confidence']:.1%} | Latency: {result['latency_ms']:.0f}ms")
else:
    print("⚠️ No videos found in data directory. Please download dataset first.")

## 4. Deploy — Chạy Streamlit App trên Local

Sau khi train xong, download thư mục `models/videomae_vsl_best/` về máy local.

```bash
# Trên máy local
git clone https://github.com/king14052004-crypto/vsl-recognition.git
cd vsl-recognition
pip install -r requirements.txt

# Copy models từ Google Drive về thư mục models/
# Hoặc dùng gdown:
# gdown --folder <drive_folder_id> -O models/

# Chạy app
streamlit run app.py
```

App sẽ:
- Mở webcam
- Buffer 16 frames (~0.5 giây ở 30fps)
- Predict mỗi 0.5 giây
- Hiển thị kết quả realtime

In [ ]:
# === Summary ===
print("=" * 60)
print("📋 DEPLOYMENT SUMMARY")
print("=" * 60)
print(f"""\

Model:          VideoMAEv2-Small (fine-tuned on Multi-VSL)
Parameters:     ~22M
Model size:     ~90 MB
Classes:        {NUM_CLASSES}
Input:          {NUM_FRAMES} frames × {IMAGE_SIZE}×{IMAGE_SIZE}

Inference:
  GPU (fp32):   ~{latencies_fp32.mean():.0f} ms ({1000/latencies_fp32.mean():.0f} FPS)
  GPU (fp16):   ~{latencies_fp16.mean():.0f} ms ({1000/latencies_fp16.mean():.0f} FPS) [recommended]
  VRAM used:    ~{torch.cuda.memory_allocated()/1e9:.1f} GB (fits 4GB GPU ✓)

Files to copy to local machine:
  models/videomae_vsl_best/
    ├── config.json
    ├── model.safetensors (or pytorch_model.bin)
    └── class_names.json

Run locally:
  streamlit run app.py
""" if torch.cuda.is_available() else f"""
Model:          VideoMAEv2-Small (fine-tuned on Multi-VSL)  
Parameters:     ~22M
Input:          {NUM_FRAMES} frames × {IMAGE_SIZE}×{IMAGE_SIZE}

Note: Running on CPU. For inference speed, use a GPU with 4GB+ VRAM.
""")